In [ ]:
# @title Install dependencies
!pip install -q -r requirements/training.txt
!pip install -q -e .
!pip install -q huggingface_hub[hf_transfer]
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
# @title Login to Hugging Face
from huggingface_hub import login
import getpass
token = getpass.getpass("Enter HF_TOKEN: ")
login(token=token)

# Clone project from HF
repo_id = "Kandil7/Baligh-1.5B"
!git clone https://huggingface.co/${repo_id} /content/Baligh 2>/dev/null || echo "Using local project"

In [ ]:
# @title Prepare Data (run once)
cd /content/Baligh
!python -m src.scripts.prepare_data --stage cpt --clean --output-dir data/train_ready
!python -m src.scripts.prepare_data --stage sft --clean --output-dir data/train_ready

In [ ]:
# @title Run CPT Training
!python -m src.scripts.run_cpt \n  --data-dir /content/Baligh/data/train_ready/cpt \n  --output-dir /content/Baligh/training/cpt \n  --eval-data /content/Baligh/data/train_ready/cpt_eval

In [ ]:
# @title Push CPT Model to HF Hub
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="/content/Baligh/training/cpt/final",
    repo_id="Kandil7/Baligh-1.5B",
    repo_type="model",
    commit_message="CPT checkpoint",
    path_in_repo="cpt"
)